# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NK0028/FlyRank_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring, implemented as binary classification whose predicted probability is used as a ranking score.**

The reviewer doesn't want a hard yes/no on one page in isolation — they want an ordered queue: "here are the top 50 pages to look at this week." That's a scoring/ranking problem. In practice the cleanest way to build it is to train a binary classifier (`needs_review`: yes/no) and use its predicted probability as the priority score, then rank by that probability and evaluate with Precision@K instead of plain accuracy. This is exactly what the Week 1-2 starter pipeline already does (random forest, Precision@50 = 0.740 vs a 0.240 hand-rule baseline) — this task extends that same framing deliberately rather than by accident.

In [1]:
# Section 1 has no numbers to check - just naming the task type.
task_type = "scoring (binary classification probability used as a ranking score)"
print("ML task type:", task_type)


ML task type: scoring (binary classification probability used as a ranking score)


## 2. Target or proxy

**Target: `needs_review` — a proxy label, not an observed ground truth.**

Defined today as: `trend_direction == "down"` AND `impressions_90d >= 100` (declining, but with enough real traffic that the decline matters). This is a **rule-defined proxy**, not something anyone directly observed — `trend_direction` itself is computed by the starter pipeline by comparing `impressions_last_30d` against `impressions_prev_30d`, a current-window comparison.

That matters: this proxy tells us a page recently declined, not that it will *keep* declining or that refreshing it will help. A stronger version of this target for later weeks would use a genuine prior-window → future-window split (e.g., features from days 1-60 predicting a decline over days 61-90), so the label reflects a real future outcome instead of a same-window rule.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

# The proxy target: rule-defined, not observed.
df["needs_review"] = ((df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)).astype(int)

print("Target source: trend_direction (same-window comparison), turned into a rule-based proxy label.")
print(df["needs_review"].value_counts(normalize=False))
print(df["needs_review"].value_counts(normalize=True).round(3))


Target source: trend_direction (same-window comparison), turned into a rule-based proxy label.
needs_review
0    16848
1    13152
Name: count, dtype: int64
needs_review
0    0.562
1    0.438
Name: proportion, dtype: float64


## 3. Success metric

**Precision@50** — of the top 50 pages the model ranks highest, what fraction are actually `needs_review == 1`?

I picked K=50 because it roughly matches a realistic weekly reviewer capacity (from the Week 1 framing: the cost of a false positive is wasted reviewer hours, so the metric should match how many pages a person can actually act on). "Good" means beating the transparent hand-rule baseline (0.240 on the starter data) by a clear margin — the Week 1-2 notebooks already showed a random forest reaching 0.740 in-sample, and roughly 0.44 under a stricter client-holdout split. Plain accuracy is not useful here: `needs_review` is a minority class, and accuracy would reward a model that just predicts "no" for everything.

In [3]:
# Section 3 has no extra numbers to check beyond what's reported above -
# this cell exists to keep the markdown-then-code rhythm consistent for 'Run All'.
metric = "Precision@50"
print("Success metric:", metric)
print("Baseline (hand rule) on this data: 0.240   |   Random forest (in-sample): 0.740   |   Random forest (client-holdout): ~0.44")


Success metric: Precision@50
Baseline (hand rule) on this data: 0.240   |   Random forest (in-sample): 0.740   |   Random forest (client-holdout): ~0.44


## 4. The unit of analysis, as a real dataframe

*One row = one content page (`content_id`), for one client, as it stands over the trailing 90-day window.*

In [4]:
unit_cols = [
    "content_id", "client_id", "content_type", "avg_position", "position_tier",
    "ctr", "impressions_90d", "days_since_last_update", "freshness_tier",
    "trend_direction", "trend_pct", "needs_review",
]
print("One row =", "one content page, for one client, over the trailing 90-day window.")
df[unit_cols].head(8)


One row = one content page, for one client, over the trailing 90-day window.


,content_id,client_id,content_type,avg_position,position_tier,ctr,impressions_90d,days_since_last_update,freshness_tier,trend_direction,trend_pct,needs_review
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,striking,0.76,3803,20,0-30,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,page_3_5,0.05,15320,25,0-30,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,page_3_5,0.09,12581,20,0-30,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,page_1,0.49,11751,22,0-30,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,page_3_5,0.13,19140,14,0-30,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,8.5,page_1,0.03,3970,20,0-30,down,-38.9,1
6,content_9a34b442b552,client_8722616204,keyword article,7.0,page_1,0.00,20,20,0-30,down,-92.3,0
7,content_a63219c6e95a,client_19581e27de,keyword article,21.2,page_3_5,0.06,1724,22,0-30,stable,0.6,0


## 5. Why ML beats a fixed rule here

A fixed if-statement can only draw straight lines through one or two signals at a time — e.g. "flag if `days_since_last_update` > 180." But `needs_review` pages don't look alike: some are old-and-stale, some are fresh-but-badly-positioned, some are high-traffic-but-low-CTR. The real pattern is a **combination** of signals (position tier, CTR, staleness, content type, age) that interact — a page with a bad `avg_position` matters differently depending on `impressions_90d`, and no single threshold captures that.

That's exactly what Week 1's numbers already showed: the hand-written rule only reached Precision@50 = 0.240, while a random forest — which can combine many weak signals nonlinearly — reached 0.740 in-sample (roughly 0.44 under an honest client-holdout split). The gap between those two numbers *is* the argument: the signal is real, but it's spread across multiple interacting columns, not sitting behind one clean threshold.

In [5]:
# Illustrate the interaction directly: the hand-rule signal alone doesn't separate
# needs_review well without also accounting for impressions_90d - a single-column rule misses this.
import numpy as np

by_staleness_alone = df.groupby(pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, 10_000]))["needs_review"].mean()
print("needs_review rate by staleness ALONE (ignoring traffic):")
print(by_staleness_alone.round(3))
print()

combo = df.groupby([pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, 10_000]),
                     pd.qcut(df["impressions_90d"], 3, duplicates="drop")])["needs_review"].mean()
print("needs_review rate by staleness x impressions_90d (the actual pattern is 2-D, not a single threshold):")
print(combo.round(3))


needs_review rate by staleness ALONE (ignoring traffic):
days_since_last_update
(-1, 30]        0.391
(30, 90]        0.514
(90, 180]       0.549
(180, 10000]    0.149
Name: needs_review, dtype: float64

needs_review rate by staleness x impressions_90d (the actual pattern is 2-D, not a single threshold):
days_since_last_update  impressions_90d     
(-1, 30]                (0.999, 205.0]          0.107
                        (205.0, 2098.333]       0.591
                        (2098.333, 517715.0]    0.571
(30, 90]                (0.999, 205.0]          0.302
                        (205.0, 2098.333]       0.630
                        (2098.333, 517715.0]    0.438
(90, 180]               (0.999, 205.0]          0.213
                        (205.0, 2098.333]       0.647
                        (2098.333, 517715.0]    0.596
(180, 10000]            (0.999, 205.0]          0.034
                        (205.0, 2098.333]       0.706
                        (2098.333, 517715.0]    1.000
N

/tmp/ipykernel_2614/2538894885.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_staleness_alone = df.groupby(pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, 10_000]))["needs_review"].mean()
/tmp/ipykernel_2614/2538894885.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  combo = df.groupby([pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, 10_000]),


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.